<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/SolarPotentiL_AND_OPEN_BUILDING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee

In [ ]:
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-jaalvalcan')

In [ ]:
# 2. Define AOI: Ikeja, Lagos, Nigeria
# Location: 6.5965° N, 3.3421° E
lagos_point = ee.Geometry.Point([3.3421, 6.5965])
aoi = lagos_point.buffer(1000).bounds()

# 3. Load Open Buildings Dataset (V3 - Polygons)
# Coverage: Africa, South/SE Asia, Latin America
buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons") \
    .filterBounds(aoi) \
    .filter(ee.Filter.gte('confidence', 0.7))

In [ ]:
# 4. Calculate Total Rooftop Area
total_roof_area = buildings.aggregate_sum('area_in_meters')

In [ ]:
# 5. Calculate Solar Potential (MW)
# 2026 Simulation Constants:
# - 200W per m2 (0.2 kW/m2)
# - 50% usable area (accounting for roof tilt and shading)
usable_ratio = 0.5
capacity_density = 0.2 # kW per m2
total_mw = ee.Number(total_roof_area).multiply(usable_ratio).multiply(capacity_density).divide(1000)

# 6. Print Results
area_val = total_roof_area.getInfo()
mw_val = total_mw.getInfo()

print(f"--- Analysis for Lagos, Nigeria ---")
print(f"Total Detected Rooftop Area: {area_val:,.2f} m²")
print(f"Estimated Solar Capacity Potential: {mw_val:,.2f} MW")

# 7. Visualization
Map = geemap.Map(center=[6.5965, 3.3421], zoom=15)
Map.addLayer(buildings, {'color': 'FF4500'}, 'Detected Buildings (Lagos)')
Map.addLayer(aoi, {'color': 'blue'}, 'Analysis AOI', False)
Map